# T1 Text-exploration notebook

Interactive exploration of the T1 next-character-prediction circuit (ARB-131 M1).

**Prereqs:** `uv sync --extra explorer` for plotly / ipywidgets / sklearn.

This notebook is a workspace, not a polished product. Each cell is something you'd want to poke at during experimentation: driving the trainer one step at a time, snapshotting interesting moments, injecting novel inputs, or mutating params to see how the circuit responds.

## Views available

| function | question it answers |
|---|---|
| `spike_raster` | which neurons fire when, and do they differ per input? |
| `sdr_overlap_matrix` | are phonetically-similar chars clustered in L2/3? |
| `weight_histograms` | are ff weights or segment perms saturated / collapsed? |
| `dendritic_segment_view` | for one L2/3 neuron, which basal segments are predictive now? |
| `pca_population` | does population structure evolve / cluster over training? |

## Setup

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

# Add repo root to sys.path so `examples.*` imports resolve from the notebook.
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / "pyproject.toml").exists():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from arbora.decoders.dendritic import DendriticDecoder
from arbora.encoders.onehot import OneHotCharEncoder
from arbora.probes.bpc import BPCProbe
from examples.text_exploration.data import (
    DEFAULT_ALPHABET,
    alphabet_filter,
    load_words,
    train_test_split,
)
from examples.text_exploration.explorer import (
    ExplorerState,
    dendritic_segment_view,
    pca_population,
    sdr_overlap_matrix,
    spike_raster,
    weight_histograms,
)
from examples.text_exploration.train import build_t1
from examples.text_exploration.trainer import T1Trainer

words = alphabet_filter(load_words(), DEFAULT_ALPHABET)
train_words, test_words = train_test_split(words, test_frac=0.2, seed=0)
print(f"{len(words)} words ({len(train_words)} train / {len(test_words)} test)")

encoder = OneHotCharEncoder(chars=DEFAULT_ALPHABET)
# `build_t1` hard-codes n_l5=0 for the M1 T1-only setup — nothing reads L5
# in this circuit so it's wasted memory + noise in the raster.
region = build_t1(encoder, seed=0)
decoder = DendriticDecoder(source_dim=region.n_l23_total, seed=0)
trainer = T1Trainer(region, encoder, decoder=decoder, bpc_probe=BPCProbe())
state = ExplorerState(trainer=trainer)
print(f"T1: cols={region.n_columns} l23={region.n_l23_total} l5={region.n_l5}")

## Quick-train

Not a lot — just enough to give the views something to show. Snapshot before and after so we can compare.

In [ ]:
state.snapshot("untrained")

for i, w in enumerate(train_words):
    state.step_word(w)
    if (i + 1) % 100 == 0:
        print(
            f"  step {i + 1}/{len(train_words)}: bpc={trainer.bpc_probe.recent_bpc:.2f}"
        )

state.snapshot("trained-1-epoch")
print(f"final bpc (recent) = {trainer.bpc_probe.recent_bpc:.3f}")

## View 1 — spike raster

Most informative view: which neurons fire when. The three laminae are stacked (L4 at top, L2/3 middle, L5 bottom), separated by red lines. Step through a word and watch the pattern.

In [ ]:
state.step_word("running", train=False)
spike_raster(state, window=20)

## View 2 — SDR overlap matrix (L4 + L2/3)

Side-by-side pairwise Jaccard for the two candidate char-representation laminae. Reset + 1 step per char, then read `l4.active` and `l23.active`.

**L4** is the input representation (driven by `ff_weights`). **L2/3** is L4 projected through `l4_to_l23_weights` — with no lateral context after reset, it's effectively L4 filtered by the projection.

- L4 collapsed, L2/3 collapsed → `ff_weights` don't differentiate chars.
- L4 distinct, L2/3 collapsed → L4→L2/3 projection is smushing them together.
- Both distinct → substrate is healthy.

Vowels go first on each axis; red lines mark the vowel/consonant divider.

In [ ]:
sdr_overlap_matrix(state)

## View 3 — weight distributions

Feedforward weights + segment permanences. Look for:

- **Saturation**: ff_weights piling up at 1.0 (too much LTP, not enough LTD).
- **Collapse**: all-zero segment perms with a narrow spike near zero (segments aren't being grown).
- **Healthy**: bimodal or broad distribution across [0, 1].

In [ ]:
weight_histograms(state)

## View 4 — dendritic segments of a chosen neuron

For a single L2/3 neuron, show every basal segment's permanences against the current prediction context.

- Red bars: source neuron active AND perm > threshold.
- Blue bars: perm > threshold but source not active.
- Gray bars: perm ≤ threshold (unused synapse).

In [ ]:
# Pick a neuron active in the most recent step so segments are actually meaningful.
import numpy as np

active = np.flatnonzero(state.trainer.region.l23.active)
neuron = int(active[0]) if active.size else 0
print(f"inspecting L2/3 neuron {neuron}")
dendritic_segment_view(state, neuron_idx=neuron)

## View 5 — PCA of population trajectory

2D PCA of all L2/3 population vectors in history. Points colored by step index. Clusters = recurring states; trajectories = sequential dynamics.

In [ ]:
pca_population(state)

## Interaction — inject + snapshot

The point of the snapshots is to let us try something (inject a novel prefix, mutate a parameter) and revert if it's a dead end.

In [ ]:
# Inject a novel prefix; watch L2/3 response without learning.
state.step_word("zxyw", train=False)
spike_raster(state, window=8)

In [ ]:
# Restore to the trained snapshot to discard any exploration above.
state.restore("trained-1-epoch")
print(f"restored; history cleared, len={len(state.history)}")

## Interaction — mutate a parameter

Toy example: kill half the L2/3 neurons (set their firing rate to 0 permanently via a column mask), then run inference. More realistic mutations would edit `ff_mask`, `perm_threshold`, `k_columns`, etc.

In [ ]:
state.snapshot("pre-mutate")
rng = np.random.default_rng(0)
kill_mask = rng.random(state.trainer.region.n_l23_total) < 0.5
# Zero out ff synapses onto killed L2/3 neurons — a coarse lesion.
ff = state.trainer.region.ff_weights
# ff_weights is (input_dim, n_l23_total-ish); column per target neuron.
ff[:, kill_mask] = 0
state.clear_history()
state.step_word("running", train=False)
spike_raster(state, window=8)

In [ ]:
state.restore("pre-mutate")
print("restored to pre-mutate")

## Interaction — ipywidgets driver

Minimal widget: type a word, click to step through it (train=False), see the raster update. Use this to sweep through prefixes without re-running cells.

In [ ]:
import ipywidgets as w
from IPython.display import display

text_in = w.Text(value="hello", description="input:")
go_btn = w.Button(description="step word", button_style="primary")
reset_btn = w.Button(description="reset region")
out = w.Output()


def _step(_):
    with out:
        out.clear_output()
        state.step_word(text_in.value, train=False)
        display(spike_raster(state, window=max(8, len(text_in.value) * 2)))


def _reset(_):
    with out:
        out.clear_output()
        state.reset()
        state.clear_history()
        print("reset")


go_btn.on_click(_step)
reset_btn.on_click(_reset)
display(w.HBox([text_in, go_btn, reset_btn]), out)